In [3]:
import pandas as pd
import numpy as np
import json
import os

print("All libraries imported successfully!")

# Check if data files exist
files = [
    '../data/docs.json',
    '../data/queries_train.json',
    '../data/queries_test.json',
    '../data/qgts_train.json'
    # if does not work use below
    # '/data/docs.json',
    # '/data/queries_train.json',
    # '/data/queries_test.json',
    # '/data/qgts_train.json'
]

print("\nchecking files...")

for file_path in files:
    if os.path.exists(file_path):
        print(f"found: {file_path}")
    else:
        print(f" MISSING: {file_path} (check your folder structure)")

All libraries imported successfully!

checking files...
found: ../data/docs.json
found: ../data/queries_train.json
found: ../data/queries_test.json
found: ../data/qgts_train.json


### data structure:
- docs.json is a library of $`questions`$ posted online that we want to search through.(creating DataFrames)

In [8]:
import json
import pandas as pd

with open("../data/docs.json", "r", encoding="UTF-8") as f:
    data = json.load(f)

# creating dataFrame(DF)
df = pd.DataFrame(data)

print(df.head(), "\n")
print(len(df))

                                            id  \
0  6970cbf2-ffff-4c7b-b73d-52524000c232_145427   
1   667220d0-66be-4059-ae60-b9df15d285f7_77850   
2   e5608645-f23d-4cf4-bda6-46071510ebb0_84546   
3  367fcce1-2ed8-4a95-81e4-77e30c1a37eb_154499   
4   75fcda1e-9b8e-4d95-bbd9-770dd6f011cc_41849   

                                                text  \
0  I try to install Complete MiKTeX 2.9 But there...   
1  I'm trying to get a launcher working for the W...   
2  Its unclear as to whether this trait means tha...   
3  I am writing a thesis with specific margin req...   
4  I always see job positions for web companies f...   

                                               title  \
0         MikTex Download Failure - toptesi.tar.lzma   
1  Launcher for a Python program that requires ex...   
2  How does the "Double damage in combat" trait w...   
3  Compiling with "latex" instead of "pdflatex" c...   
4                          Machine Learning Web Jobs   

                         

--------------------------------------------

This shows:
each row = one document 

(Pandas just displayed the first 5 rows by default)

columns: 
- id
- text
- title
- tags (list)

In [9]:
df.info()


<class 'pandas.DataFrame'>
RangeIndex: 216041 entries, 0 to 216040
Data columns (total 5 columns):
 #   Column    Non-Null Count   Dtype 
---  ------    --------------   ----- 
 0   id        216041 non-null  str   
 1   text      216041 non-null  str   
 2   title     216041 non-null  str   
 3   tags      216041 non-null  object
 4   category  216041 non-null  str   
dtypes: object(1), str(4)
memory usage: 8.2+ MB


## Explore

In [10]:
df["category"].value_counts()

category
tex            68184
unix           47382
gaming         45301
programmers    32176
android        22998
Name: count, dtype: int64

## Prepare for NLP
In NLP / search / ML, models usually expect one text input per document.
Right now, the document text is split across:

- title (short, high-signal)

- text (long, detailed)

So we merge them into one field.

In [12]:
df["full_text"] = df["title"] + " " + df["text"]


Right now, tags looks like this:
```
["gnome3", "python", "path", "cd"]
```

That’s one cell containing many values.

This is bad for:

- counting
- grouping
- filtering
- statistics

we must turn one row with a list into many rows, one per tag.

```text
Before
id	tags
A	["python", "linux"]
After
id	tags
A	python
A	linux
```

Everything else (title, text, category) is duplicated.

In [13]:
df=df.explode("tags")